In [1]:
"""
RAG Pipeline Complete Testing Notebook
=======================================
This notebook replicates ALL Phase 2 Gradio UI functionality for manual testing.

Test Coverage:
- Domain/config management (load, switch, save templates)
- Document upload & validation
- Text extraction (PDF, DOCX, TXT)
- Chunking strategies (recursive, semantic) with all parameters
- Embedding generation (sentence_transformers, Gemini)
- Vector store operations (ChromaDB, Pinecone)
- Retrieval strategies (BM25, Vector Similarity, Hybrid)
- Context assembly & prompt building
- LLM answer generation (Gemini)
- Document management (list, delete, deprecate)
"""

import os
import sys
import yaml
import json
import numpy as np
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
from datetime import datetime
from io import BytesIO
import hashlib
from typing import List, Dict, Any

# Load environment variables
load_dotenv()

# Project root setup
PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# # Core imports
from core.config_manager import ConfigManager
from core.playground_config_manager import PlaygroundConfigManager
from core.services.document_service import DocumentService
from core.pipeline.document_pipeline import DocumentPipeline

# Factory imports
from core.factories.chunking_factory import ChunkingFactory
from core.factories.embedding_factory import EmbeddingFactory
from core.factories.vectorstore_factory import VectorStoreFactory
from core.factories.retrieval_factory import RetrievalFactory

# Parser imports
from core.utils.file_parsers.parser_factory import extract_text_from_file

# Widget imports
import ipywidgets as widgets
from IPython.display import display, Markdown, HTML, clear_output

print("✅ All modules loaded successfully!")
print(f"📁 Project root: {PROJECT_ROOT}")
print(f"🔑 Gemini API Key: {'✓ Found' if os.getenv('GEMINI_API_KEY') else '✗ Not found'}")


C:\ProgramData\miniconda3\envs\myenv\lib\site-packages\pydantic\_internal\_fields.py:149: UserWarning: Field "model_name" has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
C:\ProgramData\miniconda3\envs\myenv\lib\site-packages\google\api_core\_python_version_support.py:266: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


✅ All modules loaded successfully!
📁 Project root: C:\Users\91917\Desktop\interview_preparation\Project\genai_multi_domain_platform
🔑 Gemini API Key: ✓ Found


In [2]:
# ============================================================================
# CELL 1: Setup and Imports
# ============================================================================
import sys
import yaml
from pathlib import Path
from datetime import datetime
import uuid
import logging

# Setup logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Add your project to path if needed
# sys.path.append('/path/to/your/project')

from core.playground_config_manager import PlaygroundConfigManager
from core.registry.component_registry import ComponentRegistry

print("✅ All imports successful!")
print(f"Current working directory: {Path.cwd()}")


✅ All imports successful!
Current working directory: C:\Users\91917\Desktop\interview_preparation\Project\genai_multi_domain_platform


In [3]:
# ============================================================================
# CELL 2: Verify Directory Structure
# ============================================================================
# Check if required directories exist
required_dirs = [
    "configs/playground",
    "configs/templates",
    "configs/domains",
    "configs"
]

for dir_path in required_dirs:
    path = Path(dir_path)
    exists = path.exists()
    print(f"{'✅' if exists else '❌'} {dir_path}: {'EXISTS' if exists else 'MISSING'}")
    if not exists:
        path.mkdir(parents=True, exist_ok=True)
        print(f"   → Created {dir_path}")

# Check for global_config.yaml
global_config_path = Path("configs/global_config.yaml")
if global_config_path.exists():
    print(f"\n✅ global_config.yaml found")
else:
    print(f"\n⚠️ global_config.yaml not found - will use empty defaults")


✅ configs/playground: EXISTS
✅ configs/templates: EXISTS
✅ configs/domains: EXISTS
✅ configs: EXISTS

✅ global_config.yaml found


In [4]:
# ============================================================================
# CELL 5: Create a Test Playground Config (CORRECTED)
# ============================================================================
# Generate session ID (like in app_phase2.py)
session_id = str(uuid.uuid4())[:8]
print(f"Generated Session ID: {session_id}")

# Create a sample config that matches DomainConfig schema
test_config = {
    "domain_id": "test_hr_config",  # Required
    "name": "test_hr_config",        # Required
    "display_name": "Test HR Configuration",
    "description": "Test configuration for HR documents",
    
    # vectorstore is required (not vectorstore)
    "vectorstore": {
        "provider": "chromadb",
        "chromadb": {
            "persist_directory": ".data/chromadb/test_hr",
            "collection_name": "hr_test_collection"
        }
    },
    
    "chunking": {
        "strategy": "recursive",
        "recursive": {
            "chunk_size": 500,
            "overlap": 50
        }
    },
    
    "embeddings": {
        "provider": "sentence_transformers",  # Note: underscore, not one word!
        "model_name": "all-MiniLM-L6-v2",
        "device": "cpu",
        "batch_size": 32,
        "normalize_embeddings": True
    },
    
    "retrieval": {
        "strategies": ["hybrid"],  # Must be from allowed list
        "top_k": 10,
        "similarity": "cosine",
        "enable_metadata_filtering": True,
        "normalize_scores": True,
        "hybrid": {
            "alpha": 0.7
        }
    },
    
    "security": {
        "allowed_file_types": ["pdf", "docx", "txt"],
        "max_file_size_mb": 50
    }
}

print("✅ Test config created:")
print(yaml.dump(test_config, default_flow_style=False))


Generated Session ID: f32640bb
✅ Test config created:
chunking:
  recursive:
    chunk_size: 500
    overlap: 50
  strategy: recursive
description: Test configuration for HR documents
display_name: Test HR Configuration
domain_id: test_hr_config
embeddings:
  batch_size: 32
  device: cpu
  model_name: all-MiniLM-L6-v2
  normalize_embeddings: true
  provider: sentence_transformers
name: test_hr_config
retrieval:
  enable_metadata_filtering: true
  hybrid:
    alpha: 0.7
  normalize_scores: true
  similarity: cosine
  strategies:
  - hybrid
  top_k: 10
security:
  allowed_file_types:
  - pdf
  - docx
  - txt
  max_file_size_mb: 50
vectorstore:
  chromadb:
    collection_name: hr_test_collection
    persist_directory: .data/chromadb/test_hr
  provider: chromadb



In [5]:
# ============================================================================
# CELL 4: Test ComponentRegistry (Dynamic Options)
# ============================================================================
print("Testing ComponentRegistry for dynamic UI options...\n")

# Vector stores
vectorstore_providers = ComponentRegistry.get_vectorstore_providers()
print(f"✅ Vector Store Providers: {vectorstore_providers}")

# Distance metrics
distance_metrics = ComponentRegistry.get_distance_metrics()
print(f"✅ Distance Metrics: {distance_metrics}")

# Chunking strategies
chunking_strategies = ComponentRegistry.get_chunking_strategies()
print(f"✅ Chunking Strategies: {chunking_strategies}")

# Embedding providers
embedding_providers = ComponentRegistry.get_embedding_providers()
print(f"✅ Embedding Providers: {list(embedding_providers.keys())}")
for provider, models in embedding_providers.items():
    print(f"   - {provider}: {models}")

# Device options
device_options = ComponentRegistry.get_device_options()
print(f"✅ Device Options: {device_options}")

# Retrieval strategies
retrieval_strategies = ComponentRegistry.get_retrieval_strategies()
print(f"✅ Retrieval Strategies: {retrieval_strategies}")

# LLM providers
llm_providers = ComponentRegistry.get_llm_providers()
print(f"✅ LLM Providers: {list(llm_providers.keys())}")


Testing ComponentRegistry for dynamic UI options...

✅ Vector Store Providers: ['chromadb', 'pinecone', 'faiss', 'qdrant']
✅ Distance Metrics: ['cosine', 'euclidean', 'dot_product']
✅ Chunking Strategies: ['recursive', 'semantic']
✅ Embedding Providers: ['sentence_transformers', 'openai', 'gemini']
   - sentence_transformers: ['all-MiniLM-L6-v2', 'all-mpnet-base-v2']
   - openai: ['text-embedding-ada-002']
   - gemini: ['models/embedding-001']
✅ Device Options: ['cpu', 'cuda', 'mps']
✅ Retrieval Strategies: ['hybrid', 'vector_similarity', 'bm25']
✅ LLM Providers: ['openai', 'gemini']


In [6]:
# ============================================================================
# CELL 5: Create a Test Playground Config
# ============================================================================
# Generate session ID (like in app_phase2.py)
session_id = str(uuid.uuid4())[:8]
print(f"Generated Session ID: {session_id}")

# Create a sample config (mimicking the UI form)
test_config = {
    "name": "test_hr_config",
    "description": "Test configuration for HR documents",
    "vectorstore": {
        "provider": "chromadb",
        "distance_metric": "cosine",
        "collection_name": "hr_test_collection",
        "persist_directory": "./data/chromadb/hr_test"
    },
    "chunking": {
        "strategy": "recursive",
        "recursive": {
            "chunk_size": 500,
            "overlap": 50,
            "similarity_threshold": None,
            "max_chunk_size": None
        }
    },
    "embeddings": {
        "provider": "sentence_transformers",
        "model_name": "all-MiniLM-L6-v2",
        "device": "cpu",
        "batch_size": 32
    },
    "retrieval": {
        "strategies": ["hybrid"],
        "top_k": 10,
        "hybrid": {
            "alpha": 0.7
        }
    },
    "llm_rerank": {
        "provider": "openai",
        "model_name": "gpt-3.5-turbo",
        "temperature": 0.2,
        "max_tokens": 512
    }
}

print("✅ Test config created:")
print(yaml.dump(test_config, default_flow_style=False))


Generated Session ID: d8cabd46
✅ Test config created:
chunking:
  recursive:
    chunk_size: 500
    max_chunk_size: null
    overlap: 50
    similarity_threshold: null
  strategy: recursive
description: Test configuration for HR documents
embeddings:
  batch_size: 32
  device: cpu
  model_name: all-MiniLM-L6-v2
  provider: sentence_transformers
llm_rerank:
  max_tokens: 512
  model_name: gpt-3.5-turbo
  provider: openai
  temperature: 0.2
name: test_hr_config
retrieval:
  hybrid:
    alpha: 0.7
  strategies:
  - hybrid
  top_k: 10
vectorstore:
  collection_name: hr_test_collection
  distance_metric: cosine
  persist_directory: ./data/chromadb/hr_test
  provider: chromadb



In [7]:
# ============================================================================
# CELL 6: Save Playground Config
# ============================================================================
config_name = "test_hr_config"
saved_path = PlaygroundConfigManager.save_config(config_name, session_id, test_config)

print(f"✅ Config saved to: {saved_path}")

# Verify file exists
if Path(saved_path).exists():
    print(f"✅ File verified: {saved_path}")
    with open(saved_path, 'r') as f:
        saved_content = yaml.safe_load(f)
    print(f"\nSaved config contains:")
    print(f"  - playground_name: {saved_content.get('playground_name')}")
    print(f"  - session_id: {saved_content.get('session_id')}")
    print(f"  - created_at: {saved_content.get('created_at')}")
else:
    print(f"❌ File not found: {saved_path}")


INFO:core.playground_config_manager:Saved playground config: test_hr_config_d8cabd46.yaml


✅ Config saved to: configs\playground\test_hr_config_d8cabd46.yaml
✅ File verified: configs\playground\test_hr_config_d8cabd46.yaml

Saved config contains:
  - playground_name: test_hr_config
  - session_id: d8cabd46
  - created_at: 2025-11-27T21:44:56.489584


In [8]:
# ============================================================================
# CELL 7: List All Playground Configs
# ============================================================================
all_configs = PlaygroundConfigManager.list_configs()

print(f"✅ Found {len(all_configs)} playground config(s):\n")
for idx, cfg in enumerate(all_configs, 1):
    print(f"{idx}. {cfg['name']}")
    print(f"   Filename: {cfg['filename']}")
    print(f"   Session ID: {cfg['session_id']}")
    print(f"   Created: {cfg['created_at']}")
    print()


✅ Found 1 playground config(s):

1. test_hr_config
   Filename: test_hr_config_d8cabd46.yaml
   Session ID: d8cabd46
   Created: 2025-11-27T21:44:56.489584



In [9]:
# ============================================================================
# CELL 8: Load Config by Filename
# ============================================================================
if all_configs:
    test_filename = all_configs[0]['filename']
    print(f"Loading config: {test_filename}")
    
    loaded_config = PlaygroundConfigManager.load_config(test_filename)
    
    print(f"✅ Config loaded successfully!")
    print(f"\nConfig details:")
    print(f"  Name: {loaded_config.get('name')}")
    print(f"  Description: {loaded_config.get('description')}")
    print(f"  Vector Store: {loaded_config.get('vectorstore', {}).get('provider')}")
    print(f"  Chunking Strategy: {loaded_config.get('chunking', {}).get('strategy')}")
    print(f"  Embedding Provider: {loaded_config.get('embeddings', {}).get('provider')}")
    print(f"  Retrieval Strategies: {loaded_config.get('retrieval', {}).get('strategies')}")
else:
    print("⚠️ No configs available to load")


Loading config: test_hr_config_d8cabd46.yaml

INFO:core.playground_config_manager:Loaded playground config: test_hr_config_d8cabd46.yaml



✅ Config loaded successfully!

Config details:
  Name: test_hr_config
  Description: Test configuration for HR documents
  Vector Store: chromadb
  Chunking Strategy: recursive
  Embedding Provider: sentence_transformers
  Retrieval Strategies: ['hybrid']


In [10]:
# ============================================================================
# CELL 9: Find Config by Name
# ============================================================================
search_name = "test_hr_config"
found_filename = PlaygroundConfigManager.find_config_by_name(search_name)

if found_filename:
    print(f"✅ Found config by name '{search_name}': {found_filename}")
else:
    print(f"❌ No config found with name '{search_name}'")

# Test with non-existent name
not_found = PlaygroundConfigManager.find_config_by_name("nonexistent_config")
print(f"\nSearching for 'nonexistent_config': {not_found if not_found else '❌ Not found (expected)'}")


INFO:core.playground_config_manager:Found config for 'test_hr_config': test_hr_config_d8cabd46.yaml


✅ Found config by name 'test_hr_config': test_hr_config_d8cabd46.yaml

Searching for 'nonexistent_config': ❌ Not found (expected)


In [11]:
# ============================================================================
# CELL 10: Test Merge with Global Config
# ============================================================================
print("Testing merge with global config...\n")
pg_mgr = PlaygroundConfigManager()
# Create a minimal playground config
minimal_config = {
    "name": "minimal_test",
    "vectorstore": {
        "provider": "chromadb"
    },
    "chunking": {
        "strategy": "recursive"
    }
}

# Merge with global
merged = pg_mgr.merge_with_global(minimal_config)

print(f"✅ Merged config:")
print(f"  Keys in original: {list(minimal_config.keys())}")
print(f"  Keys after merge: {list(merged.keys())}")
print(f"\nMerged config preview:")
print(yaml.dump(merged, default_flow_style=False)[:500] + "...")


Testing merge with global config...



INFO:core.playground_config_manager:Global config loaded successfully


✅ Merged config:
  Keys in original: ['name', 'vectorstore', 'chunking']
  Keys after merge: ['name', 'description', 'chunking', 'embedding', 'retrieval', 'vectorstore', 'security']

Merged config preview:
chunking:
  chunk_size: 500
  overlap: 50
  strategy: recursive
description: Global defaults for RAG system
embedding:
  api_key: ${GEMINI_API_KEY}
  mode: api_key
  model_name: gemini-1.0-pro
  project_id: ${VERTEXAI_PROJECT_ID}
  provider: Gemini
name: minimal_test
retrieval:
  filtering:
    fields:
    - author
    - doc_type
    - upload_date
    rules:
      author:
      - alice
      - bob
      doc_type:
      - policy
      - faq
      upload_date:
      - 2024-01-01:2024-12-31
  hybri...


In [12]:
# ============================================================================
# CELL 11: Save Config as Template
# ============================================================================
if all_configs:
    # Load an existing config
    source_config = PlaygroundConfigManager.load_config(all_configs[0]['filename'])
    
    # Save as template
    template_name = "test_template_hr_v1"
    template_path = Path("configs/templates") / f"{template_name}.yaml"
    
    with open(template_path, "w") as f:
        yaml.dump(source_config, f)
    
    print(f"✅ Template saved: {template_path}")
    
    # Verify
    if template_path.exists():
        print(f"✅ Template file verified")
        with open(template_path, 'r') as f:
            template_content = yaml.safe_load(f)
        print(f"\nTemplate contains {len(template_content)} top-level keys")
    else:
        print(f"❌ Template file not found")
else:
    print("⚠️ No configs available to convert to template")


INFO:core.playground_config_manager:Loaded playground config: test_hr_config_d8cabd46.yaml


✅ Template saved: configs\templates\test_template_hr_v1.yaml
✅ Template file verified

Template contains 11 top-level keys


In [13]:
# ============================================================================
# CELL 12: Create Multiple Test Configs (CORRECTED)
# ============================================================================
print("Creating multiple test configs...\n")

test_configs_data = [
    {
        "domain_id": "finance_test_config",
        "name": "finance_test_config",
        "display_name": "Finance Test Configuration",
        "description": "Finance domain test",
        "vectorstore": {
            "provider": "chromadb",
            "chromadb": {
                "persist_directory": ".data/chromadb/finance_test",
                "collection_name": "finance_test_collection"
            }
        },
        "chunking": {
            "strategy": "recursive",
            "recursive": {
                "chunk_size": 400,
                "overlap": 40
            }
        },
        "embeddings": {
            "provider": "sentence_transformers",  # Corrected
            "model_name": "all-MiniLM-L6-v2",
            "device": "cpu",
            "batch_size": 32
        },
        "retrieval": {
            "strategies": ["vector_similarity"],  # Corrected
            "top_k": 10,
            "similarity": "cosine"
        },
        "security": {
            "allowed_file_types": ["pdf", "xlsx", "txt"],
            "max_file_size_mb": 50
        }
    },
    {
        "domain_id": "legal_test_config",
        "name": "legal_test_config",
        "display_name": "Legal Test Configuration",
        "description": "Legal documents test",
        "vectorstore": {
            "provider": "chromadb",
            "chromadb": {
                "persist_directory": ".data/chromadb/legal_test",
                "collection_name": "legal_test_collection"
            }
        },
        "chunking": {
            "strategy": "recursive",
            "recursive": {
                "chunk_size": 600,
                "overlap": 60
            }
        },
        "embeddings": {
            "provider": "sentence_transformers",  # Corrected
            "model_name": "all-mpnet-base-v2",
            "device": "cpu",
            "batch_size": 32
        },
        "retrieval": {
            "strategies": ["hybrid"],  # Corrected
            "top_k": 15,
            "similarity": "cosine",
            "hybrid": {
                "alpha": 0.6
            }
        },
        "security": {
            "allowed_file_types": ["pdf", "docx"],
            "max_file_size_mb": 100
        }
    }
]

for idx, cfg_data in enumerate(test_configs_data, 1):
    session = str(uuid.uuid4())[:8]
    path = PlaygroundConfigManager.save_config(cfg_data["name"], session, cfg_data)
    print(f"{idx}. ✅ Created: {cfg_data['name']} (session: {session})")

print(f"\n✅ Total configs now: {len(PlaygroundConfigManager.list_configs())}")


Creating multiple test configs...



INFO:core.playground_config_manager:Saved playground config: finance_test_config_6d98906f.yaml
INFO:core.playground_config_manager:Saved playground config: legal_test_config_5369612f.yaml


1. ✅ Created: finance_test_config (session: 6d98906f)
2. ✅ Created: legal_test_config (session: 5369612f)

✅ Total configs now: 3


In [14]:
# ============================================================================
# CELL 13: Test Config Deletion
# ============================================================================
all_configs = PlaygroundConfigManager.list_configs()

if len(all_configs) > 1:
    # Delete the last config in the list
    to_delete = all_configs[-1]['filename']
    print(f"Attempting to delete: {to_delete}")
    
    success, message = PlaygroundConfigManager.delete_config(to_delete)
    
    print(f"\n{'✅' if success else '❌'} {message}")
    
    # Verify deletion
    remaining = PlaygroundConfigManager.list_configs()
    print(f"\nConfigs before: {len(all_configs)}")
    print(f"Configs after: {len(remaining)}")
    
    if len(remaining) == len(all_configs) - 1:
        print("✅ Deletion verified")
    else:
        print("❌ Deletion failed")
else:
    print("⚠️ Need at least 2 configs to test deletion safely")


Attempting to delete: test_hr_config_d8cabd46.yaml


INFO:core.playground_config_manager:Deleted playground config: test_hr_config_d8cabd46.yaml



✅ ✅ Deleted config 'test_hr_config_d8cabd46.yaml' (collection: hr_test_collection)

Configs before: 3
Configs after: 2
✅ Deletion verified


In [15]:
# ============================================================================
# CELL 14: Test get_service_for_config (Integration Test)
# ============================================================================
print("Testing service factory integration...\n")

def test_get_service_for_config(config_name: str):
    """Test version of get_service_for_config from app_phase2.py"""
    from core.services.document_service import DocumentService
    from core.config_manager import DomainConfig
    
    # Find playground config
    pg_filename = PlaygroundConfigManager.find_config_by_name(config_name)
    if not pg_filename:
        raise FileNotFoundError(f"Config '{config_name}' not found")
    
    # Load playground config
    pg_cfg = PlaygroundConfigManager.load_config(pg_filename)
    print(f"📄 Loaded playground config: {pg_filename}")
    print(f"   Keys: {list(pg_cfg.keys())}")
    
    # Merge with global
    merged_cfg = pg_mgr.merge_with_global(pg_cfg)
    print(f"🔀 Merged with global config")
    
    # Ensure required fields
    synth_domain_id = pg_cfg.get("playground_name") or pg_cfg.get("domain_id") or config_name
    merged_cfg.setdefault("domain_id", synth_domain_id)
    merged_cfg.setdefault("name", synth_domain_id)
    merged_cfg.setdefault("display_name", synth_domain_id)
    
    print(f"🔧 Synthetic domain_id: {synth_domain_id}")
    
    # Debug: Print what we're validating
    print(f"\n📋 Config to validate:")
    print(f"   domain_id: {merged_cfg.get('domain_id')}")
    print(f"   vectorstore: {merged_cfg.get('vectorstore', {}).get('provider')}")
    print(f"   embeddings.provider: {merged_cfg.get('embeddings', {}).get('provider')}")
    print(f"   retrieval.strategies: {merged_cfg.get('retrieval', {}).get('strategies')}")
    
    # Validate with DomainConfig
    try:
        domain_config = DomainConfig(**merged_cfg)
        print(f"\n✅ DomainConfig validated successfully!")
        print(f"   Domain ID: {domain_config.domain_id}")
        print(f"   Vector Store: {domain_config.vectorstore.provider}")
        print(f"   Chunking: {domain_config.chunking.strategy}")
        print(f"   Embeddings: {domain_config.embeddings.provider}")
        print(f"   Retrieval: {domain_config.retrieval.strategies}")
        return domain_config
    except Exception as e:
        print(f"\n❌ Validation failed: {e}")
        print(f"\n🔍 Debugging info:")
        print(f"   Check if your config matches DomainConfig schema requirements")
        print(f"   Common issues:")
        print(f"   - Use 'sentence_transformers' not 'sentence_transformers'")
        print(f"   - Use 'vectorstore' not 'vectorstore'")
        print(f"   - Strategies must be from: ['vector_similarity', 'hybrid', 'bm25']")
        print(f"   - Required fields: domain_id, name, vectorstore")
        raise

# Test with existing config
all_configs = PlaygroundConfigManager.list_configs()
if all_configs:
    test_config_name = all_configs[0]['name']
    print(f"🧪 Testing with config: {test_config_name}\n")
    print("=" * 70)
    try:
        domain_cfg = test_get_service_for_config(test_config_name)
        print("\n" + "=" * 70)
        print("✅ SERVICE FACTORY TEST PASSED!")
        print("=" * 70)
    except Exception as e:
        print("\n" + "=" * 70)
        print(f"❌ SERVICE FACTORY TEST FAILED")
        print("=" * 70)
        import traceback
        traceback.print_exc()
else:
    print("⚠️ No configs available for testing")


INFO:core.playground_config_manager:Found config for 'legal_test_config': legal_test_config_5369612f.yaml


Testing service factory integration...

🧪 Testing with config: legal_test_config



INFO:core.playground_config_manager:Loaded playground config: legal_test_config_5369612f.yaml


📄 Loaded playground config: legal_test_config_5369612f.yaml
   Keys: ['domain_id', 'name', 'display_name', 'description', 'vectorstore', 'chunking', 'embeddings', 'retrieval', 'security', 'playground_name', 'session_id', 'created_at', 'last_modified']
🔀 Merged with global config
🔧 Synthetic domain_id: legal_test_config

📋 Config to validate:
   domain_id: legal_test_config
   vectorstore: chromadb
   embeddings.provider: sentence_transformers
   retrieval.strategies: ['hybrid']

✅ DomainConfig validated successfully!
   Domain ID: legal_test_config
   Vector Store: chromadb
   Chunking: recursive
   Embeddings: sentence_transformers
   Retrieval: ['hybrid']

✅ SERVICE FACTORY TEST PASSED!


In [16]:
# ============================================================================
# CELL 15: Test Cleanup Expired Configs
# ============================================================================
import time

print("Testing cleanup of expired configs...\n")

# Create a test config and manually set old timestamp
old_config = {
    "name": "old_test_config",
    "description": "This should be cleaned up",
    "created_at": "2023-01-01T00:00:00"  # Old date
}

old_session = str(uuid.uuid4())[:8]
old_path = PlaygroundConfigManager.save_config("old_test_config", old_session, old_config)

# Manually override the created_at to make it old
with open(old_path, 'r') as f:
    old_cfg = yaml.safe_load(f)
old_cfg['created_at'] = "2023-01-01T00:00:00"
with open(old_path, 'w') as f:
    yaml.dump(old_cfg, f)

print(f"✅ Created old config: {old_path}")

# Run cleanup (expire configs older than 1 hour for testing)
before_count = len(PlaygroundConfigManager.list_configs())
deleted_count = PlaygroundConfigManager.cleanup_expired_configs(expiry_hours=1)
after_count = len(PlaygroundConfigManager.list_configs())

print(f"\nCleanup results:")
print(f"  Configs before: {before_count}")
print(f"  Deleted: {deleted_count}")
print(f"  Configs after: {after_count}")

if deleted_count > 0:
    print(f"✅ Cleanup working correctly")
else:
    print(f"⚠️ No configs deleted (may need to adjust expiry_hours)")


Testing cleanup of expired configs...



INFO:core.playground_config_manager:Saved playground config: old_test_config_463d4d69.yaml


✅ Created old config: configs\playground\old_test_config_463d4d69.yaml


INFO:core.playground_config_manager:Cleaned up expired config: old_test_config_463d4d69.yaml (age: 25485.7h)
INFO:core.playground_config_manager:Cleanup completed: 1 configs deleted



Cleanup results:
  Configs before: 3
  Deleted: 1
  Configs after: 2
✅ Cleanup working correctly


In [ ]:
# ============================================================================
# CELL 18: Optional Cleanup (Run if you want to remove test data)
# ============================================================================
import shutil

cleanup = input("Do you want to clean up all test files? (yes/no): ")

if cleanup.lower() == 'yes':
    # Remove all playground test configs
    # for file in Path("configs/playground").glob("*.yaml"):
    #     if "test" in file.name.lower() or "workflow" in file.name.lower():
    #         file.unlink()
    #         print(f"🗑️ Deleted: {file}")
    
    # # Remove test templates
    # for file in Path("configs/templates").glob("*.yaml"):
    #     if "test" in file.name.lower() or "workflow" in file.name.lower():
    #         file.unlink()
    #         print(f"🗑️ Deleted: {file}")
    
    print("\n✅ Cleanup completed!")
else:
    print("Skipping cleanup - test files preserved")


In [ ]:
%run jupyter_test_playground.py